# Feature Engineering & Merging Demo
This notebook demonstrates how to merge datasets and apply feature engineering using the `DataMerger` and `FeatureEngineer` classes.

In [ ]:
import os
import sys
import pandas as pd

# Change working directory to project root so paths line up
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

if '..' not in sys.path:
    sys.path.append('..')

from src.stats_transformer.featurization.data_merger import DataMerger
from src.stats_transformer.featurization.feature_engineering import FeatureEngineer

## 1. Merging Data files
Let's load two datasets from `data/panel/monthly` and merge them.

In [ ]:
merger = DataMerger(params_path='params.yaml')

df_credit = merger.load_dataset('data/panel/monthly/bis_credit_panel.parquet')
df_cpi = merger.load_dataset('data/panel/monthly/oecd_cpi_panel.parquet')

print('BIS Credit Shape:', df_credit.shape)
print('OECD CPI Shape:', df_cpi.shape)

# Note: data must be merged on entity & date. Data Mergers expects country, date.
# First check columns
print('\nCredit Columns:', df_credit.columns.tolist())
print('CPI Columns:', df_cpi.columns.tolist())

In [ ]:
# Standardize entities to 'country' and merge
df_credit = merger.standardize_entity(df_credit, 'country')
df_cpi = merger.standardize_entity(df_cpi, 'country')

df_merged = merger.merge(df_credit, df_cpi, on=['country', 'date'], how='inner')
print('\nMerged Data Shape:', df_merged.shape)
display(df_merged.head())

## 2. Feature Engineering
Now we instantiate a `FeatureEngineer` and apply some transformations to our numeric columns. For this example, let's say we want a rolling mean and percent change.

In [ ]:
# We can pass transformations manually:
numeric_cols = df_merged.select_dtypes(include=['number']).columns.tolist()
data_columns = [col for col in numeric_cols if col not in ['country', 'date']]

engineer = FeatureEngineer(
    params_path=None,
    transformations=['changepct', 'rollingmean'],
    entity_column='country',
    date_column='date',
    period='monthly',
    data_columns=data_columns,
    verbose=True
)

df_features = engineer.fit_transform(df_merged)

print('\nFeaturized Data Shape:', df_features.shape)
display(df_features.head())